#### The transformer revolution started with a simple question:  
**Why don’t we feed the entire input sequence?**

![Transformers Intuition](Images/Transformers_intuition.png)

---

This process is called **tokenization**, and it's the first of three steps before feeding input into the model.  

Instead of treating input as a sequence, we convert it into a **set of tokens**.

> If we are talking about sets, it means the order is irrelevant.

---

### Word Embeddings

After tokenization, we project words into a **vector space** (word embeddings):

1. Words are strongly correlated with each other.  
   → When projected into a continuous Euclidean space, we can capture semantic relationships.

2. Next, we introduce a clever trick to restore the notion of order.

---

### Positional Encodings

When you convert a sequence into a set (via tokenization), you lose the notion of order.

Since transformers process inputs as sets, they are **permutation invariant**.

To fix this, we add **positional encodings**:

- These are small constants added to word embeddings before the first self-attention layer.
- They inject information about the position of each token.
- As a result, the same word appearing in different positions gets slightly different representations.

![Positional Embedding](Images/Positional_embedding.png)

---

### Comparison with Recurrent Models

This is in contrast to **recurrent models**:

- RNNs inherently preserve order.
- However, they struggle to capture long-range dependencies (i.e., attending to distant tokens).

### Intution behind Query, Keys and Values matrices in Transformers

In practice, the Transformer uses 3 different representations: the Queries, Keys and Values of the embedding matrix, a good way to think about this is, When you search (query) for a particular video, the search engine will map your query against a set of keys (video title, description, etc.) associated with possible stored videos. Then the algorithm will present you the best-matched videos (values). This is the foundation of content/feature-based lookup.

![Queries Keys Values intution](Images/Q_K_V_Intution.png)



# 🧠 Self-Attention in Transformers

## What is Self-Attention?

Self-attention enables a model to find relationships between different words in an input sequence, capturing both **syntactic** and **contextual** dependencies.

---

## 📌 Intuition

Consider the input sentence:

> **"Hello I love you"**

A trained self-attention layer will assign:
- Higher importance (weights) between **"love" → "I"** and **"love" → "you"**
- Lower importance to **"Hello"**

This happens because:
- "I", "love", and "you" form a **subject–verb–object** relationship
- The model learns these linguistic patterns automatically

---

## 🖼️ Self-Attention Visualization

![Self Attention](Images/Self_Attention.png)

---

## 🔁 Skip Connections in Transformers

Just like in CNNs, Transformers also use **skip (residual) connections**.

### Why are they important?
- Help in better **gradient flow**
- Allow the model to retain **original input information**
- Enable deeper architectures without degradation

---

## 📏 Layer Normalization

Since Transformers operate on vectors, **Layer Normalization** is applied.

### What does it do?
- Normalizes activations across features (channels)
- Stabilizes training
- Computes:
  - Mean
  - Variance

---

## ⚙️ MLP (Feed-Forward Network)

On top of the attention block, Transformers include a **Multi-Layer Perceptron (MLP)**.

### Structure:
- Two linear (fully connected) layers
- Non-linear activation (e.g., ReLU / GELU)
- Dropout for regularization

---

### 🤔 What does the MLP do?

- Introduces **non-linearity** into the model
- Learns **complex feature transformations**
- Helps build **richer representations**

> ❗ Without the MLP, the model would struggle to learn complex patterns.

---

## 🖼️ Transformer Encoder Block

![Transformer Encoder](Images/Transformer_encoder.png)

In [ ]:
'''
MLP Intuition, this is the linear block in the Transformer Encoder.
'''

import torch.nn as nn 
dim = 512 
dim_linear_block = 4 * dim 

mlp = nn.Sequential(
    nn.Linear(dim, dim_linear_block), '''Expanding the dimension'''
    nn.ReLU(),
    nn.Dropout(0.1),
    nn.Linear(dim_linear_block, dim), ''' Shrinking the dimension back to the original dimension'''
    nn.Dropout(0.1)
)

### Processing a Sentence in a Transformer -- The Encoder Part !! 

To process a sentence, we follow **three key steps**:

---

#### 1. Word Embeddings
- Compute embeddings for all words in the input sentence **simultaneously**.

---

#### 2. Positional Encoding
- Apply positional encodings to each embedding.
- This results in **word vectors that capture both meaning and position**, if a same word appears twice due to positional encoding the transformers understand they might be referring to a different thing/context.

---

#### 3. Input to Encoder
- The resulting word vectors are passed to the **first encoder block**.

---

### Encoder Block Architecture

Each encoder block consists of the following components (in order):

1. **Multi-Head Self-Attention**
   -Find correlations between all pairs of words in a sentence.

2. **Add & Normalize**
   - Residual connection + normalization

3. **Feedforward Network (MLP)**
   - Two linear layers with a non-linear activation in between

4. **Add & Normalize**
   - Second residual connection + normalization

---

### 🧠 Summary Flow

```text
Input → Embedding → Positional Encoding → Encoder Block

Encoder Block:
    → Multi-Head Attention
    → Add & Norm
    → Feedforward (MLP)
    → Add & Norm

### 🧠 Transformer Decoder Block

The **Transformer Decoder** contains all the components of the encoder, along with two additional layers:

- **Masked Multi-Head Attention**
- **Encoder–Decoder Attention (Multi-Head Attention Layer)**

![Encoder Decoder Transformer](Images/Encoder%20Decoder%20Transformer.png)

---

### 🔒 What is Masked Multi-Head Attention?

During the **decoding stage**, the model generates output **one word at a time**.

This introduces an important constraint:

> The model should only attend to **previous words**, not future ones.

To enforce this, we use **Masked Multi-Head Attention**, which prevents the model from "seeing" future tokens.

#### 💡 Why is masking needed?

If the model had access to the full sentence during training, it could "cheat" by looking ahead. Masking ensures:

- At time step *t*, the model only uses tokens from `1 → t`
- Future tokens are **hidden (masked out)**

---

### 📥 Input to the Decoder

The decoder does **not** receive the target sentence as-is. Instead, it receives a **shifted version** of the target.

#### Example:

| Type              | Sequence            |
|-------------------|-------------------|
| Target            | `I love you`       |
| Input to Decoder  | `<start> I love`   |

So at each step, the decoder predicts the **next word**.

---

### 🔁 What Happens Inside the Decoder?

Let’s say we are translating:

- Input (Encoder): `"Je t’aime"`
- Output (Decoder): `"I love you"`

At a given step:

#### 1. Masked Self-Attention (Decoder Side)

- Query comes from: `"I love"`
- The model asks:  
  👉 *"What have I generated so far?"*

---

#### 2. Encoder–Decoder Attention

- Query: `"I love"` (decoder side)
- Key & Value: Encoder output (`"Je t’aime"`)

The model learns alignment:

- `"I"` ↔ `"Je"`
- `"love"` ↔ `"aime"`

👉 This helps the decoder focus on relevant parts of the input sequence.

---

#### 3. Feed Forward Network (MLP)

- Combines:
  - Context from decoder (past words)
  - Context from encoder (input sentence)

👉 Outputs the probability distribution for the **next word**

---

### 🔄 Intuition: What Decoder Does at Each Step

At every time step, the decoder performs:

1. **Masked Attention**  
   → *"What have I said so far?"*

2. **Encoder Attention**  
   → *"What was the input?"*

3. **MLP (Feed Forward Layer)**  
   → *"Given both, what should I say next?"*

---

### 🧩 Summary

- Masking ensures **causal generation** (no future leakage)
- Decoder input is **shifted right**
- Combines:
  - Past generated tokens
  - Encoder's understanding of input
- Predicts **next token step-by-step**

---

💡 Think of the decoder as a **controlled generator** that:
- remembers what it has said,
- looks back at the input,
- and decides what to say next.